# Lección 2.2 — Calidad de datos: faltantes, atípicos, duplicados y ruido

> **Módulo 2 · Los datos: el combustible** · Nivel 1 — Cero
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 2.1 (tipos de datos)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Diagnosticar los cuatro problemas clásicos de calidad: **faltantes**, **atípicos**, **duplicados** y **ruido**.
- Elegir una estrategia para los datos faltantes (eliminar, imputar, marcar) y explicar sus riesgos.
- Distinguir un atípico que es **un error** de uno que es **un caso raro pero real**.
- Unificar categorías escritas de formas distintas y encontrar duplicados escondidos.
- Explicar por qué el ruido en las etiquetas daña más a los modelos muy flexibles.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: basura entra, basura sale

Un chef excelente con ingredientes echados a perder cocina un plato malo. No hay técnica que lo salve.

En ML se dice **"basura entra, basura sale"** (*garbage in, garbage out*). Un modelo aprende **lo que hay** en los datos, incluidos sus errores. Si un sistema registró la edad de un cliente como 250 años, el modelo intentará aprender algo de ese cliente imposible.

En la práctica, los científicos de datos pasan **más tiempo limpiando datos que entrenando modelos**. Vamos a conocer a los cuatro sospechosos habituales.

### ✋ Ejercicio 1 — Detective de datos
Un gimnasio te pasa esta tabla de socios. Sin ejecutar nada, encuentra **al menos 4 problemas distintos**.

In [ ]:
socios = pd.DataFrame({
    "nombre":  ["Ana", "Luis", "Marta", "Luis", "Pedro", "Sofía", "Raúl"],
    "edad":    [34, 28, 250, 28, None, 41, 37],
    "sede":    ["Norte", "Centro", "norte", "Centro", "Sur ", "Centro", "Norte"],
    "visitas_mes": [12, 8, 15, 8, 3, None, 10],
    "renovo":  ["sí", "no", "sí", "no", "no", "sí", "sí"],
})
socios

✍️ *Problemas que encontré:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Faltantes:** la edad de Pedro y las visitas de Sofía están vacías (`NaN`/`None`).
2. **Atípico imposible:** Marta tiene **250 años**. Seguramente un error de captura (¿25?).
3. **Duplicado:** Luis aparece dos veces con exactamente los mismos datos.
4. **Categorías inconsistentes:** "Norte" y "norte" son la misma sede; "Sur " tiene un espacio al final. Para la computadora son **tres sedes distintas** escritas de forma diferente.

Si encontraste los cuatro, ya piensas como alguien que limpia datos. 🕵️
</details>

---
## 2. 📖 Datos faltantes: los huecos de la tabla

Un **dato faltante** es una celda vacía. En pandas aparece como `NaN` (*Not a Number*) o `None`. Casi todos los modelos de scikit-learn **se niegan a entrenar** si encuentran uno.

### ¿Por qué falta un dato? (importa mucho)
| Tipo | Ejemplo | ¿Peligroso? |
|---|---|---|
| **Al azar** | Una encuesta en papel se mojó y se borraron algunas casillas | Poco: los huecos no tienen patrón |
| **Depende de otras columnas** | Los jóvenes contestan menos la pregunta de ingresos | Medio: se puede compensar con esas otras columnas |
| **Depende del propio valor** | Quienes ganan mucho **prefieren no decir** su ingreso | Mucho: el hueco **es** información |

> 💡 A veces que un dato falte **dice algo**. Un cliente que no llenó su teléfono quizás está menos comprometido. Por eso, a veces conviene **guardar una marca** de que el dato faltaba.

### Estrategia 1: eliminar filas
Es lo más fácil: si una fila tiene algún hueco, se borra. Pero puede salir muy caro.

### 🔮 Ejercicio 2 — ¿Cuántas filas sobreviven?
Tenemos 1,000 clientes y 5 columnas. En **cada** columna falta, al azar, el **10 %** de los datos. Si borramos toda fila que tenga **al menos un** hueco, ¿cuántas filas quedan aproximadamente?

- a) ~900 · b) ~800 · c) ~600 · d) ~500

✍️ *Mi apuesta:*

In [ ]:
rng = np.random.default_rng(0)
clientes = pd.DataFrame(rng.normal(50, 10, size=(1000, 5)), columns=["a", "b", "c", "d", "e"])
# Borramos al azar el 10 % de cada columna
for col in clientes.columns:
    clientes.loc[rng.random(1000) < 0.10, col] = np.nan

print("Huecos por columna:\n", clientes.isna().sum(), sep="")
print("\nFilas originales            :", len(clientes))
print("Filas tras borrar con huecos:", len(clientes.dropna()))

<details><summary><b>👀 Ver respuesta</b></summary>

**c) ~600.** Cada fila tiene un 90 % de probabilidad de estar completa en *cada* columna, así que la probabilidad de estar completa en las 5 es 0.9 × 0.9 × 0.9 × 0.9 × 0.9 ≈ **0.59**. Perdimos ~40 % de los datos aunque cada columna solo tenía un 10 % de huecos.

Con 20 columnas quedaría ~12 %. **Borrar filas es razonable solo cuando los huecos son pocos.**
</details>

### Estrategia 2: imputar (rellenar)
**Imputar** es rellenar el hueco con un valor razonable:
- **Numéricos:** la **media** (promedio) o la **mediana** (el valor del medio al ordenar).
- **Categóricos:** la categoría **más frecuente**, o una categoría nueva "Desconocido".

¿Media o mediana? Depende de si hay valores extremos.

### 🔮 Ejercicio 3 — El millonario en la oficina
Cinco empleados ganan entre 12,000 y 16,000 al mes. Entra el dueño, que gana 2,000,000. Vamos a rellenar el salario faltante de un sexto empleado. Antes de ejecutar: ¿qué valor parece más "típico", la media o la mediana?

In [ ]:
salarios = pd.Series([12_000, 15_000, 14_000, 13_000, 16_000, 2_000_000, np.nan],
                     index=["Ana", "Beto", "Caro", "Dani", "Eva", "Dueño", "Nuevo"])

print(f"Media   : {salarios.mean():>12,.0f}")
print(f"Mediana : {salarios.median():>12,.0f}")

<details><summary><b>👀 Ver respuesta</b></summary>

La **mediana** (14,500) es mucho más representativa. La media (≈345,000) está **arrastrada** por un solo valor extremo: nadie en la oficina, salvo el dueño, gana algo parecido.

**Regla práctica:** si la columna tiene valores extremos o está "cargada" hacia un lado (ingresos, precios, tiempos), imputa con la **mediana**.
</details>

### Estrategia 3: imputar **y** marcar
Scikit-learn tiene `SimpleImputer`, que rellena y, si se lo pides, agrega una columna que dice **"aquí había un hueco"** (`add_indicator=True`). Así el modelo puede aprender si faltar el dato es, en sí, una señal.

In [ ]:
tabla = pd.DataFrame({"ingreso": [20_000, np.nan, 35_000, 18_000, np.nan],
                      "edad":    [25, 40, np.nan, 31, 52]})

imputador = SimpleImputer(strategy="median", add_indicator=True)   # ✏️ prueba strategy="mean"
rellena = imputador.fit_transform(tabla)

pd.DataFrame(rellena, columns=imputador.get_feature_names_out())

### ✋ Ejercicio 4
1. ¿Con qué valor se rellenaron los ingresos faltantes? ¿De dónde salió?
2. ¿Qué significan las dos columnas nuevas que empiezan con `missingindicator_`?

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Con **20,000**: la mediana de los ingresos conocidos (18,000, 20,000 y 35,000).
2. Valen **1** donde el dato faltaba y **0** donde estaba. Si los clientes que no reportan ingresos se comportan distinto, el modelo podrá notarlo.

> ⚠️ **Muy importante (lo verás en la Lección 2.6):** el imputador *aprende* la mediana de los datos que le das. Si se la calculas con **todos** los datos, incluidos los de prueba, estás haciendo trampa sin darte cuenta. Siempre se aprende **solo con el entrenamiento**.
</details>

---
## 3. 📖 Atípicos (*outliers*): ¿error o caso raro?

Un **atípico** es un valor muy distinto del resto. Hay dos tipos, y se tratan de forma opuesta:

| | Error | Caso raro pero real |
|---|---|---|
| **Ejemplo** | Edad = 250 · Precio = −5 · Altura = 17 m | Un cliente que gasta 100 veces más que el promedio · Un terremoto |
| **Qué hacer** | Corregirlo si se puede; si no, tratarlo como **faltante** | **Conservarlo** (¡a veces es justo lo que quieres detectar!) |

Una forma común de **detectar** candidatos es la regla del **rango intercuartílico (IQR)**, que es lo que dibuja un diagrama de caja: se marca como sospechoso todo lo que queda muy lejos de la "caja" donde vive el 50 % central de los datos.

> 💡 La regla **señala** sospechosos; no decide. Decidir si es un error o un caso real requiere **conocer el negocio**.

In [ ]:
edades = pd.Series([23, 35, 41, 29, 38, 250, 33, 45, 27, 31, 52, -3, 36, 40, 88])

q1, q3 = edades.quantile([0.25, 0.75])
iqr = q3 - q1
limite_bajo, limite_alto = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Rango 'normal' según la regla IQR: de {limite_bajo:.1f} a {limite_alto:.1f}")
print("Sospechosos:", edades[(edades < limite_bajo) | (edades > limite_alto)].tolist())

plt.figure(figsize=(3, 4))
plt.boxplot(edades); plt.xticks([]); plt.ylabel("edad")
plt.title("Diagrama de caja: los puntos sueltos son los sospechosos"); plt.show()

### ✋ Ejercicio 5 — Juez de atípicos
La regla señaló tres valores. Para cada uno decide: ¿**error** o **caso real**? ¿Qué harías?

✍️ *Mis decisiones:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **250 → error** (nadie vive 250 años). Tratarlo como faltante e imputarlo, o corregirlo si puedes consultar la fuente.
- **−3 → error** (no hay edades negativas). Igual que el anterior.
- **88 → probablemente real.** La regla lo señala porque el grupo es joven, pero una persona de 88 años existe. **Conservarlo.** Borrarlo sería ocultarle al modelo que existen clientes mayores.

Esta es la lección clave: **la estadística señala, el conocimiento del negocio decide.**
</details>

### 📖 ¿Por qué importan los atípicos?
Algunos modelos se dejan arrastrar mucho por un solo punto extremo. La **regresión lineal** (Lección 5.1), que busca la recta que mejor pasa por los puntos, es un ejemplo clásico.

### 🔮 Ejercicio 6 — Un solo punto
Tenemos 20 casas: a más metros cuadrados, más precio. Alguien capturó **una** casa de 60 m² con precio de 90 millones (se le fueron tres ceros de más). Antes de ejecutar: ¿cuánto crees que cambiará la pendiente de la recta (cuánto sube el precio por cada m² extra)?
- a) Casi nada · b) Un poco · c) Muchísimo, incluso podría cambiar de signo

In [ ]:
rng = np.random.default_rng(3)
m2 = rng.uniform(50, 150, 20)
precio = 20_000 * m2 + rng.normal(0, 150_000, 20)          # ≈ $20,000 por m²

m2_error = np.append(m2, 60)
precio_error = np.append(precio, 90_000_000)               # ← el error de captura

limpio = LinearRegression().fit(m2.reshape(-1, 1), precio)
sucio = LinearRegression().fit(m2_error.reshape(-1, 1), precio_error)
print(f"Pendiente SIN el error: {limpio.coef_[0]:>10,.0f} por m²")
print(f"Pendiente CON el error: {sucio.coef_[0]:>10,.0f} por m²")

x = np.linspace(45, 155, 2).reshape(-1, 1)
plt.scatter(m2, precio / 1e6, label="casas reales")
plt.plot(x, limpio.predict(x) / 1e6, "g-", label="recta sin el error")
plt.plot(x, sucio.predict(x) / 1e6, "r--", label="recta con el error")
plt.ylim(0, 5); plt.xlabel("m²"); plt.ylabel("precio (millones)"); plt.legend(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

**c)** Un solo punto (de 21) cambia la pendiente de **≈ +20,000** a un valor **negativo**: ¡con el error, el modelo concluye que las casas más grandes son **más baratas**! La recta roja se inclina hacia el punto absurdo, que ni siquiera cabe en la gráfica.

Otros modelos son más resistentes (los árboles, Lección 5.4, casi no se inmutan), pero **ningún** modelo aprende bien de datos con errores grandes.
</details>

---
## 4. 📖 Duplicados y categorías inconsistentes

### Duplicados
Una fila repetida (por un error de exportación, un formulario enviado dos veces…) hace que el modelo le dé **doble importancia** a ese ejemplo. Y si una copia termina en entrenamiento y otra en prueba, el modelo "ya vio el examen" (fuga de datos, Lección 2.6).

### Categorías inconsistentes
"CDMX", "cdmx", "Ciudad de México" y "CDMX " (con espacio) son, para la computadora, **cuatro ciudades distintas**. Esto parte los datos de un mismo grupo y además **esconde duplicados**.

### 🔮 Ejercicio 7 — Duplicados escondidos
Una tienda registra pedidos. Antes de ejecutar: `duplicated()` busca filas **exactamente iguales**. ¿Cuántos duplicados encontrará antes de limpiar el texto? ¿Y después?

✍️ *Mi apuesta:* antes ___ · después ___

In [ ]:
ventas = pd.DataFrame({
    "cliente": ["Ana López", "ana lópez", "Beto Ruiz", "Ana López ", "Caro Díaz", "Beto Ruiz"],
    "ciudad":  ["CDMX", "cdmx", "Monterrey", "CDMX", "Ciudad de México", "monterrey"],
    "monto":   [450, 450, 1200, 450, 300, 1200],
})
print("Duplicados exactos ANTES de limpiar:", ventas.duplicated().sum())
print("Ciudades distintas ANTES:", ventas["ciudad"].nunique(), ventas["ciudad"].unique().tolist())

# Limpieza: quitar espacios sobrantes, pasar a minúsculas y unificar nombres
ventas["cliente"] = ventas["cliente"].str.strip().str.lower()
ventas["ciudad"] = ventas["ciudad"].str.strip().str.lower().replace({"ciudad de méxico": "cdmx"})

print("\nDuplicados exactos DESPUÉS de limpiar:", ventas.duplicated().sum())
print("Ciudades distintas DESPUÉS:", ventas["ciudad"].nunique(), ventas["ciudad"].unique().tolist())

<details><summary><b>👀 Ver respuesta</b></summary>

**Antes: 0 duplicados.** Ninguna fila es *idéntica* letra por letra: cambian las mayúsculas o hay un espacio al final. Y aparecen 5 "ciudades" distintas.

**Después: 3 duplicados** (Ana López aparece 3 veces y Beto Ruiz 2: hay 3 copias de más) y solo **2 ciudades**: `cdmx` y `monterrey` (la "Ciudad de México" de Caro también era `cdmx`).

**Orden correcto:** primero **unificar el texto**, después **buscar duplicados**. Al revés, se te escapan.

> ⚠️ Ojo: ¿y si Ana de verdad hizo tres compras iguales de 450? Para saberlo necesitarías una fecha o un número de pedido. Antes de borrar duplicados, pregúntate si **pueden ser reales**.
</details>

### ✏️ Ejercicio 8 — Termina la limpieza
Ejecuta la celda para borrar los duplicados. Luego ✏️ cambia `keep="first"` por `keep=False` y vuelve a ejecutar **las dos celdas anteriores**. ¿Qué hace `keep=False`? ¿Por qué casi nunca es lo que quieres?

In [ ]:
sin_duplicados = ventas.drop_duplicates(keep="first")    # ✏️ prueba keep=False
print("Filas antes:", len(ventas), "· después:", len(sin_duplicados))
sin_duplicados

<details><summary><b>👀 Ver respuesta</b></summary>

Con `keep="first"` quedan **3 filas**: se conserva la primera aparición de cada pedido. Con `keep=False` se borran **todas** las copias, incluida la original: solo sobrevive Caro (1 fila). Perderías los pedidos de Ana y Beto por completo.
</details>

---
## 5. 📖 Ruido: cuando los datos mienten un poco

**Ruido** es el error aleatorio que tienen todos los datos reales: una báscula que marca 200 g de más, un encuestado que contesta al azar, un médico que etiqueta mal una radiografía.
- **Ruido en las características:** las mediciones son imprecisas.
- **Ruido en las etiquetas:** la "respuesta correcta" está mal (un correo normal marcado como *spam*). Este es especialmente dañino.

El ruido **no se puede eliminar del todo**. Lo importante es que el modelo **no lo memorice**.

### 🔮 Ejercicio 9 — ¿Quién sufre más el ruido?
Vamos a "ensuciar" a propósito un porcentaje de las etiquetas de entrenamiento (cambiar 0 por 1 y viceversa) y medir la exactitud en datos de prueba **limpios**. Compararemos dos modelos:
- Un **árbol sin límites**, muy flexible: puede memorizar cada ejemplo (Lección 5.4).
- Una **regresión logística**, muy simple: solo traza una línea recta (Lección 5.2).

¿Cuál crees que pierde más exactitud a medida que sube el ruido?

✍️ *Mi apuesta:*

In [ ]:
X, y = make_classification(n_samples=2000, n_features=10, n_informative=5, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

rng = np.random.default_rng(0)
print("ruido   árbol   logística")
for ruido in [0.0, 0.1, 0.2, 0.3, 0.4]:            # ✏️ prueba otros porcentajes
    y_ruidoso = y_tr.copy()
    voltear = rng.random(len(y_tr)) < ruido
    y_ruidoso[voltear] = 1 - y_ruidoso[voltear]
    arbol = DecisionTreeClassifier(random_state=0).fit(X_tr, y_ruidoso)
    logistica = LogisticRegression().fit(X_tr, y_ruidoso)
    print(f"{ruido:>4.0%}   {arbol.score(X_te, y_te):.1%}   {logistica.score(X_te, y_te):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

El **árbol sin límites** sufre mucho más: cae unos 35 puntos (de ≈89 % a ≈54 %, casi como lanzar una moneda) al llegar a 40 % de ruido, porque **memoriza** cada etiqueta equivocada. La **logística** casi no se inmuta hasta el 30 % y apenas cae unos 4 puntos con 40 %: una línea recta no puede doblarse para acomodar cada error, así que los errores "se compensan".

Esto no significa que la logística sea siempre mejor (con datos limpios y patrones complejos el árbol puede ganar). Significa que **cuanto más flexible es un modelo, más fácilmente memoriza el ruido**: es el sobreajuste (Lección 4.4), visto desde los datos.
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué hacer |
|---|---|---|
| `ValueError: Input contains NaN` | Datos faltantes | Imputar (`SimpleImputer`) o eliminar con criterio |
| Perdiste medio conjunto de datos con `dropna()` | Huecos repartidos en muchas columnas | Imputar en lugar de borrar |
| Promedios absurdos o rectas que cambian de signo | Atípicos que son errores | Revisarlos; tratar los errores como faltantes |
| Borraste "atípicos" y el modelo ya no detecta fraudes | Eliminaste casos raros **reales** | Conservar los casos reales; borrar solo errores |
| Más categorías de las que existen | Mayúsculas, espacios, sinónimos | `.str.strip().str.lower()` y un diccionario de equivalencias |
| El modelo acierta "demasiado" | Duplicados repartidos entre entrenamiento y prueba | Quitar duplicados **antes** de dividir (Lección 2.6) |

### 🔮 Ejercicio 10 — El "0" sospechoso
Un sensor de temperatura de una bodega envía `0` cuando se queda sin batería. Su tabla tiene 1,000 lecturas y **ningún** `NaN`. ¿Está libre de faltantes?

In [ ]:
rng = np.random.default_rng(5)
temperatura = pd.Series(rng.normal(18, 2, 1000).round(1))
temperatura[rng.random(1000) < 0.08] = 0          # el sensor sin batería

print("NaN reportados por pandas:", temperatura.isna().sum())
print("Lecturas exactamente en 0 :", (temperatura == 0).sum())
print(f"Media 'ingenua'           : {temperatura.mean():.1f} °C")
print(f"Media sin los ceros falsos: {temperatura.replace(0, np.nan).mean():.1f} °C")

<details><summary><b>👀 Ver respuesta</b></summary>

**No.** Hay 83 lecturas en `0` que en realidad son **faltantes disfrazados**. Pandas no las ve como huecos y la media "ingenua" baja de 18.0 °C a 16.5 °C. Valores como `0`, `-1`, `999`, `"N/A"` o `"?"` se usan a menudo para decir "no sé". **Conocer cómo se capturaron los datos** es la única forma de descubrirlos; luego se convierten en `NaN` con `.replace(...)`.
</details>

---
## 7. 🧠 Resumen en 5 puntos
1. **Basura entra, basura sale:** un modelo aprende también los errores de los datos.
2. **Faltantes:** borrar filas sale caro; **imputar** con la mediana (numéricos) o la moda (categóricos) y **marcar** el hueco suele ser mejor. Cuidado con los faltantes disfrazados (`0`, `999`, `"N/A"`).
3. **Atípicos:** la estadística los **señala**, el negocio **decide**. Los errores se corrigen o se tratan como faltantes; los casos raros reales se conservan.
4. **Duplicados y categorías:** primero **unifica el texto**, después busca duplicados.
5. **Ruido:** siempre existe; los modelos muy flexibles lo **memorizan**.

---
## 8. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 8.1 Repaso rápido (opción múltiple)

**1.** Una columna de ingresos tiene algunos valores enormes y un 5 % de huecos. ¿Con qué conviene rellenarla?
- a) Con la media · b) Con la mediana · c) Con 0 · d) Con el valor máximo

**2.** Tienes 50 columnas y en cada una falta un 2 % al azar. Si borras toda fila con algún hueco, aproximadamente conservarás…
- a) 98 % · b) 90 % · c) 64 % · d) 36 %

**3.** Un banco quiere detectar transacciones fraudulentas, que son montos rarísimos y muy altos. ¿Qué haces con esos atípicos?
- a) Borrarlos porque distorsionan el modelo · b) Conservarlos: son justo lo que buscas · c) Reemplazarlos por la mediana · d) Convertirlos en faltantes

**4.** ¿Cuál es el orden correcto?
- a) Buscar duplicados → unificar texto · b) Unificar texto → buscar duplicados · c) Da igual

**5.** ¿Qué modelo memoriza más fácilmente las etiquetas equivocadas?
- a) Una regresión logística · b) Un árbol de decisión sin límites · c) Los dos igual

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** La mediana no se deja arrastrar por los valores enormes. Rellenar con 0 inventaría clientes sin ingresos.
2. **d)** 0.98 multiplicado 50 veces ≈ 0.36. Con muchas columnas, hasta pocos huecos destruyen la tabla.
3. **b)** Son casos raros **reales** y son el objetivo. Borrarlos dejaría al modelo ciego.
4. **b)** Si buscas duplicados antes, "Ana" y "ana " no parecen iguales.
5. **b)** El árbol sin límites puede crear una hoja para cada ejemplo, incluidos los mal etiquetados.
</details>

### 8.2 Preguntas para pensar (sin ejecutar)

**A.** En una encuesta de salud, la pregunta "¿cuántas bebidas alcohólicas tomas por semana?" tiene muchos huecos. ¿Por qué podría ser **peligroso** rellenarlos simplemente con la mediana? ¿Qué harías en su lugar?

**B. Caso de análisis 💼** Una aseguradora detecta que el 3 % de sus pólizas tienen la edad del conductor en `0`. Un compañero propone borrar esas filas "porque son pocas". ¿Qué preguntas harías antes de aceptar?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Probablemente los huecos **no son al azar**: quien bebe mucho puede preferir no contestar. Rellenar con la mediana los convierte a todos en "bebedores moderados" y **borra** justo la señal importante. Mejor: imputar **y agregar un indicador** de "no contestó", para que el modelo pueda aprender si no contestar se asocia con el resultado.

**B.** ¿Por qué hay ceros? ¿Es un valor por defecto del sistema (faltante disfrazado)? ¿Esas pólizas tienen algo en común (un canal de venta, una sucursal, un periodo)? Si sí, borrarlas **sesgaría** el modelo contra ese grupo. ¿Se puede recuperar la edad de otra fuente (fecha de nacimiento en la licencia)? Lo más prudente suele ser tratarlas como faltantes, imputar y marcar.
</details>

### 8.3 🧩 Práctica integradora: "La encuesta del gimnasio"

Una cadena de gimnasios encuestó a sus socios para entender quién **renueva** su membresía. Te entrega la tabla tal cual salió del sistema. Tu trabajo: **dejarla limpia**.

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
rng = np.random.default_rng(42)
n = 300
gimnasio = pd.DataFrame({
    "id_socio":    np.arange(1001, 1001 + n),
    "edad":        rng.normal(35, 10, n).round().clip(16, 75),
    "sede":        rng.choice(["Norte", "Centro", "Sur"], n),
    "visitas_mes": rng.poisson(9, n).astype(float),
    "satisfaccion": rng.integers(1, 6, n).astype(float),    # de 1 a 5
})
gimnasio["renovo"] = (rng.random(n) < 0.3 + 0.05 * gimnasio["visitas_mes"].clip(0, 12)).astype(int)
# --- El sistema "ensucia" los datos (no cambies estas líneas) ---
gimnasio.loc[rng.choice(n, 12, replace=False), "edad"] = np.nan
gimnasio.loc[rng.choice(n, 5, replace=False), "edad"] = [0, 150, 220, -1, 0]
gimnasio.loc[rng.choice(n, 20, replace=False), "visitas_mes"] = np.nan
gimnasio.loc[rng.choice(n, 30, replace=False), "satisfaccion"] = 99      # 99 = "no contestó"
gimnasio = pd.concat([gimnasio, gimnasio.sample(15, random_state=1)], ignore_index=True)
desorden = {"Norte": ["norte", "NORTE", "Norte "], "Centro": ["centro", "Centro "], "Sur": ["sur", "SUR"]}
for i in rng.choice(len(gimnasio), 45, replace=False):
    gimnasio.loc[i, "sede"] = rng.choice(desorden[gimnasio.loc[i, "sede"]])

print("Filas:", len(gimnasio))
gimnasio.drop(columns="id_socio").describe().round(1)

**Paso 2 ✋ — Diagnóstico.** Mirando el resumen de arriba (`describe`), ¿qué dos columnas tienen valores **imposibles**? Luego ejecuta la celda de abajo para contar el resto de los problemas.

In [ ]:
print("Faltantes por columna:\n", gimnasio.isna().sum(), sep="")
print("\nDuplicados exactos:", gimnasio.duplicated().sum())
print("\nSedes:\n", gimnasio["sede"].value_counts(), sep="")

<details><summary><b>👀 Ver respuesta</b></summary>

- **`edad`**: mínimo −1 y máximo 220 (también hay ceros). **`satisfaccion`**: máximo 99 en una escala de 1 a 5 → es el código de "no contestó", un **faltante disfrazado**.
- Faltantes: 11 en `edad` y 22 en `visitas_mes`.
- Duplicados exactos: **10**. Guarda ese número para el paso siguiente.
- Sedes: **10** variantes para solo 3 sedes reales ("Centro" y "Centro " parecen iguales en pantalla, pero el segundo tiene un espacio al final).
</details>

**Paso 3 ✏️ — Unifica las sedes y luego quita duplicados** (¡en ese orden!). Completa la línea marcada usando lo que aprendiste en la Sección 4.

In [ ]:
gimnasio["sede"] = gimnasio["sede"].str.strip().str.title()     # ✏️ ¿qué hace .title()? Pruébalo
print("Sedes:", gimnasio["sede"].unique().tolist())

antes = len(gimnasio)
gimnasio = gimnasio.drop_duplicates()
print(f"Filas: {antes} → {len(gimnasio)}")

<details><summary><b>👀 Ver respuesta</b></summary>

`.str.title()` pone la primera letra en mayúscula ("NORTE" → "Norte"). Quedan 3 sedes y se eliminan **15** duplicados: 315 → **300** filas.

¡En el Paso 2 solo veías **10**! Los otros 5 eran **duplicados escondidos**: la misma fila con la sede escrita distinto ("Sur" en una copia y "SUR" en la otra). Por eso **siempre** se unifica el texto primero.
</details>

**Paso 4 ✏️ — Convierte los valores imposibles en faltantes.** Decide qué rango de edad es posible para un socio de gimnasio y cambia los límites si quieres.

In [ ]:
EDAD_MIN, EDAD_MAX = 14, 100     # ✏️

imposible = (gimnasio["edad"] < EDAD_MIN) | (gimnasio["edad"] > EDAD_MAX)
print("Edades imposibles:", gimnasio.loc[imposible, "edad"].tolist())
gimnasio.loc[imposible, "edad"] = np.nan
gimnasio["satisfaccion"] = gimnasio["satisfaccion"].replace(99, np.nan)

print("\nFaltantes ahora:\n", gimnasio.isna().sum(), sep="")

<details><summary><b>👀 Ver respuesta</b></summary>

Se marcan como faltantes las 5 edades imposibles (0, 150, −1, 0, 220) y los 30 "99" de satisfacción. Ahora hay **16** faltantes en `edad`, **20** en `visitas_mes` y **30** en `satisfaccion`. (Con otros límites de edad, tus números pueden variar un poco.)
</details>

**Paso 5 🔮 — ¿Borrar o imputar?** Antes de ejecutar: si borras toda fila con algún hueco, ¿qué porcentaje de las 300 filas crees que conservarás?

✍️ *Mi apuesta:*

In [ ]:
print(f"Filas completas: {len(gimnasio.dropna())} de {len(gimnasio)} ({len(gimnasio.dropna()) / len(gimnasio):.0%})")

<details><summary><b>👀 Ver respuesta</b></summary>

Conservarías **236 de 300 (79 %)**: perderías una de cada cinco encuestas. Y ojo: los 30 socios que **no contestaron** la satisfacción podrían ser justo los más descontentos. Borrarlos quitaría información valiosa. Mejor imputar y marcar.
</details>

**Paso 6 ▶️ — Imputa y marca.** Rellenamos las numéricas con la mediana y agregamos indicadores. ¿Cuántas columnas tiene el resultado y por qué?

In [ ]:
numericas = ["edad", "visitas_mes", "satisfaccion"]
imputador = SimpleImputer(strategy="median", add_indicator=True)
limpias = pd.DataFrame(imputador.fit_transform(gimnasio[numericas]),
                       columns=imputador.get_feature_names_out(), index=gimnasio.index)

print("Medianas usadas:", dict(zip(numericas, imputador.statistics_)))
print("Columnas:", limpias.columns.tolist())
print("¿Quedan huecos?", limpias.isna().sum().sum())

<details><summary><b>👀 Ver respuesta</b></summary>

**6 columnas**: las 3 originales ya sin huecos y 3 indicadores `missingindicator_...` (uno por cada columna que tenía faltantes). Las medianas usadas son 34.5 años, 9 visitas y 3 de satisfacción.

🎉 La tabla está lista para un modelo. Pero fíjate: hicimos `fit_transform` con **toda** la tabla. En un proyecto real, **primero se divide** en entrenamiento y prueba (Lección 2.5), y el imputador aprende las medianas **solo** del entrenamiento (Lección 2.6). Lo harás bien en el proyecto del módulo.
</details>

---
## 📚 Para profundizar
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Imputation of missing values*.
- 📘 **Documentación:** pandas, guía *Working with missing data*.
- 📺 **Video:** StatQuest, "Boxplots" y "Quantiles and Percentiles" (inglés, con subtítulos), para entender la regla del IQR.

---
⬅️ **Anterior:** [2.1 Tipos de datos: no todos los ingredientes se cocinan igual](2.1-tipos-de-datos.ipynb) · ➡️ **Siguiente:** [2.3 Preparación: escalado y codificación de categorías](2.3-preparacion-escalado-y-codificacion.ipynb)